In [ ]:
from wotan import flatten
import matplotlib.pyplot as plt
import pandas as pd
from astropy.timeseries import LombScargle
import numpy as np
from astropy.timeseries import BoxLeastSquares
import lightkurve as lk

In [ ]:
# cargar datos
file = pd.read_csv("tic_427393298_s_98_corrected.csv")

#print(file.columns)
time = file["time"]
flux = file["flux"]
print(time,flux)

# plot 
plt.scatter(time, flux, s=5)

In [ ]:
def explore_detrend(window_length, break_tol):

    flat, trend = flatten(
        time,
        flux,
        window_length=window_length,
        break_tolerance=break_tol,
        method="hspline",
        return_trend=True
    )

    plt.figure(figsize=(12,5))
    plt.scatter(time, flux, s=3, color='k', label="Original")
    plt.plot(time, trend, color='r', label="Trend")
    plt.legend()
    plt.title(f"window={window_length}, break={break_tol}")
    plt.show()

    return flat, trend

In [ ]:
flat_flux, trend_flux = explore_detrend(.6, 1.0)

df = pd.DataFrame({
    "time": time,          
    "flat_flux": flat_flux,
    "trend_flux": trend_flux
})
t0_B = time[np.argmin(flat_flux)]
print(t0_B)
df.to_csv("TIC_199780530_s_6_trending.csv", index=False)

In [ ]:

def iterative_transit_search(time, flux, max_signals=5):

    residuals = flux.copy()
    detected = []

    for n in range(max_signals):

        print(f"\n--- Buscando señal {n+1} ---")

        # quitar NaNs
        mask_valid = np.isfinite(residuals)
        t_clean = time[mask_valid]
        f_clean = residuals[mask_valid]

        bls = BoxLeastSquares(t_clean, f_clean)

        periods = np.linspace(0.5, 20, 10000)
        duration = 0.1

        power = bls.power(periods, duration)

        best_idx = np.argmax(power.power)
        P = periods[best_idx]
        t0 = power.transit_time[best_idx]

        print(f"Periodo detectado: {P:.4f} d")

        # =========================
        #  evitar duplicados
        # =========================
        if any(abs(P - p)/p < 0.05 for p, _ in detected):
            print("Periodo ya detectado, saltando...")
            continue

        # =========================
        # detectar alias P/2
        # =========================
        if any(abs(P - 2*p)/p < 0.05 or abs(2*P - p)/p < 0.05 for p, _ in detected):
            print("⚠️ Posible alias (P vs P/2)")

        # =========================
        # modelo box
        # =========================
        model = bls.model(time, P, duration, t0)

        # =========================
        # masking 
        # =========================
        in_transit = model < (np.median(model) - 1e-4)

        residuals = residuals.copy()
        residuals[in_transit] = np.nan

        # guardar
        detected.append((P, t0))

        print(f"Señales detectadas hasta ahora: {len(detected)}")

    return detected, residuals

In [ ]:
detected, residuals = iterative_transit_search(time, flux)
P_detected = detected[0][0]# estamos haciendo un diccionario, entonces primerp cuargamos en detected dos valores, pero a su vez el return nos da dos entonces accedemos a detected con [0]y luego a la posicion en que se guarda todo en detected
t0_detected = detected[0][1]

print("P_detected:", P_detected)

In [ ]:
def plot_phase(time, flux, P):

    # 🔹 fase centrada en primario
    t0 = time[np.argmin(flux)]
    phase = ((time - t0) % P) / P - 0.5

    # 🔹 duplicar (evita cortes)
    phase = np.concatenate([phase, phase + 1])
    flux = np.concatenate([flux, flux])

    # 🔹 primario
    i1 = np.argmin(flux)
    p1, f1 = phase[i1], flux[i1]

    # 🔹 secundario (lejos del primario)
    mask = np.abs(phase - p1) > 0.2
    i2 = np.argmin(flux[mask])
    p2, f2 = phase[mask][i2], flux[mask][i2]

    # 🔹 baseline
    baseline = np.median(flux)

    # 🔹 plot
    plt.figure(figsize=(10,4))
    plt.scatter(phase, flux, s=3, color='k')

    # marcar eclipses
    plt.axvline(p1, color='red', ls='--')
    plt.axvline(p2, color='blue', ls='--')

    # profundidades (opcional pero útil)
    plt.vlines(p1, f1, baseline, color='red')
    plt.vlines(p2, f2, baseline, color='blue')

    plt.xlim(-0.5, 1.5)
    plt.xlabel("Phase")
    plt.ylabel("Flux")
    plt.title(f"P = {P_real:.4f}")
    plt.grid(alpha=0.3)
    
P_real = P_detected
plot_phase(time, flat_flux, P_real*2)

In [ ]:
def get_rotation_period(time, flux_flat, P_orb,
                        eclipse_width=0.08,
                        min_period=0.1,
                        max_period=20,
                        plot=True):

    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.timeseries import LombScargle

    # =========================
    # 🔹 asegurar numpy arrays
    # =========================
    time = np.asarray(time)
    flux_flat = np.asarray(flux_flat)

    # =========================
    # 🔹 fase orbital (para masking)
    # =========================
    t0 = time[np.argmin(flux_flat)]
    phase = ((time - t0) % P_orb) / P_orb - 0.5

    # =========================
    # 🔹 máscara de eclipses
    # =========================
    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # =========================
    # 🔹 datos limpios
    # =========================
    time_clean = time[~mask_eclipse]
    flux_clean = flux_flat[~mask_eclipse]

    mask_valid = np.isfinite(flux_clean)
    time_clean = time_clean[mask_valid]
    flux_clean = flux_clean[mask_valid]

    # =========================
    # 🔹 normalizar
    # =========================
    flux_clean = flux_clean / np.median(flux_clean)

    # =========================
    # 🔹 Lomb-Scargle
    # =========================
    freq, power = LombScargle(time_clean, flux_clean).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period
    )

    periods = 1 / freq

    # =========================
    # 🔹 periodo candidato
    # =========================
    Prot = periods[np.argmax(power)]

    # =========================
    # 🔥 chequeo simple P vs 2P
    # =========================
    def scatter_metric(P):
        ph = (time_clean % P) / P
        return np.std(ph)

    if scatter_metric(2 * Prot) < scatter_metric(Prot):
        print("⚠️ Usando 2P (alias detectado)")
        Prot = 2 * Prot

    # =========================
    # 🔹 plots
    # =========================
    if plot:

        # -------- periodograma --------
        plt.figure(figsize=(8,4))
        plt.plot(periods, power, color='k')
        plt.axvline(Prot, color='red', ls='--')
        plt.xlabel("Period [days]")
        plt.ylabel("Power")
        plt.title("Rotation Period")
        plt.xlim(0, min(max_period, 10))
        plt.grid(alpha=0.3)

        # -------- FASE ROTACIÓN --------
        phase_rot = (time_clean % Prot) / Prot - 0.5
        phase_rot_plot = np.concatenate([phase_rot, phase_rot + 1])
        flux_rot_plot = np.concatenate([flux_clean, flux_clean])

        # -------- FASE ORBITAL --------
        phase_orb = ((time - t0) % P_orb) / P_orb - 0.5
        phase_orb_plot = np.concatenate([phase_orb, phase_orb + 1])
        flux_orb_plot = np.concatenate([flux_flat, flux_flat])

        # -------- PLOT FUSIONADO --------
        plt.figure(figsize=(9,5))

        # orbital (fondo)
        plt.scatter(
            phase_orb_plot,
            flux_orb_plot,
            s=5,
            alpha=0.25,
            color='C1',
            label=f"Orbital (P={P_orb:.3f} d)"
        )

        # rotación (encima)
        plt.scatter(
            phase_rot_plot,
            flux_rot_plot,
            s=5,
            alpha=0.5,
            color='C0',
            label=f"Rotation (P={Prot:.3f} d)"
        )

        plt.xlabel("Folded Phase")
        plt.ylabel("Normalized Flux")
        plt.title("Orbital vs Rotational Modulation")

        plt.xlim(-0.5, 1.5)

        # límites robustos
        all_flux = np.concatenate([flux_rot_plot, flux_orb_plot])
        plt.ylim(np.percentile(all_flux, 1),
                 np.percentile(all_flux, 99))

        plt.grid(alpha=0.3)
        plt.legend()

    # =========================
    # 🔹 output
    # =========================
    print("\n--- RESULTADOS ---")
    print(f"P_orb = {P_orb:.4f} d")
    print(f"P_rot = {Prot:.4f} d")
    print(f"P_rot / P_orb = {Prot/P_orb:.3f}")

    return Prot, periods, power, time_clean, flux_clean

In [ ]:
P_real = P_detected

Prot, periods, power, time_clean, flux_clean = get_rotation_period(
    time,
    flat_flux,
    P_real,
    plot=True
)

In [ ]:
def compute_width(p_center):

    if p_center is None:
        return None, None, None

    # distancia circular
    dist = np.abs(phase - p_center)
    dist = np.minimum(dist, 1 - dist)

    mask = dist < 0.25
    ph = phase[mask]
    fl = flux[mask]

    # centrar en 0
    ph = ph - p_center
    ph[ph > 0.5] -= 1
    ph[ph < -0.5] += 1

    # grid de anchos
    widths = np.linspace(0.01, 0.25, 150)

    best_chi2 = np.inf
    best = None

    for w in widths:

        inside = np.abs(ph) < w/2
        outside = ~inside

        if np.sum(inside) < 5 or np.sum(outside) < 5:
            continue

        f_in = np.mean(fl[inside])
        f_out = np.mean(fl[outside])

        model = np.where(inside, f_in, f_out)

        chi2 = np.sum((fl - model)**2)

        if chi2 < best_chi2:
            best_chi2 = chi2
            best = (w, f_in, f_out)

    if best is None:
        return None, None, None

    w_best, f_in, f_out = best

    phi_start = p_center - w_best/2
    phi_end   = p_center + w_best/2

    return w_best, phi_start, phi_end

In [ ]:
results = analyze_eclipses(time, flat_flux, P_real*2)

In [ ]:
# =========================
# 1. Cargar datos
# =========================

time = file["time"].values
flux = file["flux"].values

# =========================
# 2. Limpieza básica
# =========================
mask = np.isfinite(time) & np.isfinite(flux)
time = time[mask]
flux = flux[mask]

# =========================
# 3. LightCurve
# =========================
lc = lk.LightCurve(time=time, flux=flux)

# =========================
# 4. Detrending
# =========================
lc_flat = lc.flatten(window_length=201)
lc_flat = lc_flat.remove_outliers(sigma=5)

# =========================
# 5. BLS → PERIODO ORBITAL
# =========================
bls = lc_flat.to_periodogram(
    method='bls',
    minimum_period=0.5,
    maximum_period=10,
    frequency_factor=5
)

P_orb = bls.period_at_max_power.value
t0 = bls.transit_time_at_max_power.value

print(f"Periodo orbital (BLS): {P_orb:.5f} días")
print(f"t0: {t0:.5f}")

# =========================
# 6. PROFUNDIDAD (ROBUSTA)
# =========================
mask_transit = bls.get_transit_mask(
    period=P_orb,
    transit_time=bls.transit_time_at_max_power,
    duration=bls.duration_at_max_power
)

flux_in = lc_flat.flux.value[mask_transit]
flux_out = lc_flat.flux.value[~mask_transit]

F_in = np.median(flux_in)
F_out = np.median(flux_out)

depth = 1 - (F_in / F_out)

print(f"Profundidad (global): {depth:.6f}")
print(f"Rp/Rs ≈ {np.sqrt(depth):.4f}")

# =========================
# 7. FASEADO (P y 2P)
# =========================
lc_fold = lc_flat.fold(period=P_orb, epoch_time=t0)
lc_fold_2 = lc_flat.fold(period=2*P_orb, epoch_time=t0)

# =========================
# 8. DETECTAR PRIMARIO / SECUNDARIO
# =========================
phase = lc_fold_2.phase.value
flux_phase = lc_fold_2.flux.value

# baseline global
F_out_phase = np.median(flux_phase)

# ventanas aproximadas
mask_primary = (phase > -0.1) & (phase < 0.1)
mask_secondary = (phase > 0.4) & (phase < 0.6)

F_primary = np.median(flux_phase[mask_primary])
F_secondary = np.median(flux_phase[mask_secondary])

depth_primary = 1 - (F_primary / F_out_phase)
depth_secondary = 1 - (F_secondary / F_out_phase)

print("\n--- Binaria eclipsante ---")
print(f"Profundidad primaria:   {depth_primary:.6f}")
print(f"Profundidad secundaria: {depth_secondary:.6f}")

# =========================
# 9. PLOTS
# =========================

# --- BLS ---
bls.plot()
plt.title("BLS Periodogram")
plt.show()

# --- Fase P ---
lc_fold.scatter()
plt.title("Fase con P")
plt.show()

# --- Fase 2P ---
plt.figure()
plt.scatter(phase, flux_phase, s=3)

plt.axhline(F_out_phase, color='green', label='Baseline')
plt.axhline(F_primary, color='red', label='Primario')
plt.axhline(F_secondary, color='orange', label='Secundario')

plt.legend()
plt.title("Fase con 2P (binaria eclipsante)")
plt.xlabel("Fase")
plt.ylabel("Flux")
plt.show()

# =========================
# 10. REMOVER ECLIPSES
# =========================
lc_no_transit = lc_flat[~mask_transit]

t = lc_no_transit.time.value
f = lc_no_transit.flux.value

mask = np.isfinite(t) & np.isfinite(f)
t = t[mask]
f = f[mask]

# =========================
# 11. LOMB-SCARGLE (ROTACIÓN)
# =========================
min_period = 0.2
max_period = 20

frequency, power = LombScargle(t, f).autopower(
    minimum_frequency=1/max_period,
    maximum_frequency=1/min_period
)

best_frequency = frequency[np.argmax(power)]
P_rot = 1 / best_frequency

print(f"\nPeriodo de rotación: {P_rot:.5f} días")

# =========================
# 12. PLOT LS
# =========================
plt.figure()
plt.plot(1/frequency, power)
plt.xlabel("Periodo (días)")
plt.ylabel("Power")
plt.title("Lomb-Scargle")
plt.gca().invert_xaxis()
plt.show()

# =========================
# 13. FASE ROTACIONAL
# =========================
phase_rot = (t % P_rot) / P_rot

plt.figure()
plt.scatter(phase_rot, f, s=2)
plt.xlabel("Fase")
plt.ylabel("Flux")
plt.title("Modulación rotacional")
plt.show()

In [ ]:
# ============================================================
# TIC 427393298
# ROTATIONAL + ORBITAL ANALYSIS PER TESS SECTOR
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import LombScargle
from scipy.signal import savgol_filter

# ============================================================
# CONFIG
# ============================================================

P_ORB = 3.9050  # <-- cambia aquí tu periodo orbital real

# ============================================================
# FILES
# ============================================================

files = [
    "tic 427393298_s_6_corrected.csv",
    "tic_427393298_s_32_corrected.csv",
    "tic_427393298_s_98_corrected.csv"
]

sector_labels = {
    "tic 427393298_s_6_corrected.csv": "S6",
    "tic_427393298_s_32_corrected.csv": "S32",
    "tic_427393298_s_98_corrected.csv": "S98"
}
# ============================================================
# LOAD DATA
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(fname)

    print(f"\nColumns in {fname}:")
    print(df.columns)

    # --------------------------------------------------------
    # detectar columnas automáticamente
    # --------------------------------------------------------

    time_col = None
    flux_col = None

    for c in df.columns:

        cl = c.lower()

        if time_col is None and "time" in cl:
            time_col = c

        if flux_col is None and (
            "flux" in cl
            or "corr" in cl
            or "flat" in cl
        ):
            flux_col = c

    if time_col is None:
        raise ValueError("No TIME column found")

    if flux_col is None:
        raise ValueError("No FLUX column found")

    print(f"Using TIME column: {time_col}")
    print(f"Using FLUX column: {flux_col}")

    time = np.array(df[time_col])
    flux = np.array(df[flux_col])

    # --------------------------------------------------------
    # limpiar NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    return time, flux


# ============================================================
# DETRENDING
# ============================================================

def detrend_flux(time, flux,
                 window=301,
                 polyorder=2):

    flux = flux / np.nanmedian(flux)

    trend = savgol_filter(
        flux,
        window_length=window,
        polyorder=polyorder
    )

    flat_flux = flux / trend

    return flat_flux, trend


# ============================================================
# ROTATION PERIOD
# ============================================================

def get_rotation_period(time,
                        flux_flat,
                        P_orb,
                        eclipse_width=0.08,
                        min_period=0.1,
                        max_period=20):

    # --------------------------------------------------------
    # eclipse masking
    # --------------------------------------------------------

    t0 = time[np.argmin(flux_flat)]

    phase = ((time - t0) % P_orb) / P_orb - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    time_clean = time[~mask_eclipse]
    flux_clean = flux_flat[~mask_eclipse]

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time_clean) & np.isfinite(flux_clean)

    time_clean = time_clean[mask]
    flux_clean = flux_clean[mask]

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux_clean = flux_clean / np.median(flux_clean)

    # --------------------------------------------------------
    # Lomb-Scargle
    # --------------------------------------------------------

    freq, power = LombScargle(
        time_clean,
        flux_clean
    ).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period
    )

    periods = 1 / freq

    # --------------------------------------------------------
    # best period
    # --------------------------------------------------------

    Prot = periods[np.argmax(power)]

    return Prot, periods, power, time_clean, flux_clean


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_sector(time,
                flux,
                flat_flux,
                trend,
                Prot,
                periods,
                power,
                sector):

    fig = plt.figure(figsize=(14,9))

    # ========================================================
    # RAW LIGHT CURVE
    # ========================================================

    ax1 = plt.subplot(221)

    ax1.plot(
        time,
        flux,
        '.k',
        ms=2,
        alpha=0.5,
        label="Raw Flux"
    )

    ax1.plot(
        time,
        trend,
        color='red',
        lw=2,
        label="Trend"
    )

    ax1.set_title(f"{sector} — Raw Light Curve")

    ax1.set_xlabel("Time [BTJD]")
    ax1.set_ylabel("Flux")

    ax1.legend()

    # ========================================================
    # DETRENDED
    # ========================================================

    ax2 = plt.subplot(222)

    ax2.plot(
        time,
        flat_flux,
        '.k',
        ms=2
    )

    ax2.set_title(f"{sector} — Detrended")

    ax2.set_xlabel("Time [BTJD]")
    ax2.set_ylabel("Normalized Flux")

    # ========================================================
    # PERIODOGRAM
    # ========================================================

    ax3 = plt.subplot(223)

    ax3.plot(
        periods,
        power,
        color='k'
    )

    ax3.axvline(
        Prot,
        color='red',
        ls='--',
        lw=2,
        label=f"P_rot = {Prot:.3f} d"
    )

    ax3.axvline(
        P_ORB,
        color='C1',
        ls=':',
        lw=2,
        label=f"P_orb = {P_ORB:.3f} d"
    )

    ax3.set_xlim(0, 10)

    ax3.set_xlabel("Period [days]")
    ax3.set_ylabel("Power")

    ax3.set_title("Lomb-Scargle Periodogram")

    ax3.legend()

    # ========================================================
    # ROTATIONAL PHASE
    # ========================================================

    ax4 = plt.subplot(224)

    phase_rot = (time % Prot) / Prot

    phase_plot = np.concatenate([
        phase_rot,
        phase_rot + 1
    ])

    flux_plot = np.concatenate([
        flat_flux,
        flat_flux
    ])

    ax4.scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5
    )

    ax4.set_xlim(0, 2)

    ax4.set_xlabel("Rotational Phase")
    ax4.set_ylabel("Normalized Flux")

    ax4.set_title(
        f"P_rot / P_orb = {Prot/P_ORB:.2f}"
    )

    plt.tight_layout()

    plt.show()


# ============================================================
# MAIN LOOP
# ============================================================

results = []

for fname in files:

    print("\n===================================================")
    print(f"PROCESSING: {fname}")

    sector = sector_labels[fname]

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    time, flux = load_sector_file(fname)

    # --------------------------------------------------------
    # DETREND
    # --------------------------------------------------------

    flat_flux, trend = detrend_flux(
        time,
        flux
    )

    # --------------------------------------------------------
    # ROTATION
    # --------------------------------------------------------

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB
    )

    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    results.append({
        "sector": sector,
        "P_rot": Prot,
        "P_orb": P_ORB,
        "ratio": Prot / P_ORB
    })

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print("\nRESULTS")
    print(f"P_orb = {P_ORB:.4f} d")
    print(f"P_rot = {Prot:.4f} d")
    print(f"P_rot / P_orb = {Prot/P_ORB:.3f}")

    # --------------------------------------------------------
    # PLOT
    # --------------------------------------------------------

    plot_sector(
        time,
        flux,
        flat_flux,
        trend,
        Prot,
        periods,
        power,
        sector
    )

# ============================================================
# RESULTS TABLE
# ============================================================

results = pd.DataFrame(results)

print("\n===================================================")
print("FINAL RESULTS")
print(results)

# ============================================================
# EVOLUTION OF ROTATION
# ============================================================

plt.figure(figsize=(7,4))

sector_num = [
    int(s.replace("S",""))
    for s in results["sector"]
]

plt.plot(
    sector_num,
    results["P_rot"],
    '-o',
    lw=2
)

plt.axhline(
    P_ORB,
    color='red',
    ls='--',
    label='P_orb'
)

plt.xlabel("TESS Sector")
plt.ylabel("Rotation Period [days]")

plt.title("Evolution of Stellar Rotation")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

# ============================================================
# PHASE COMPARISON
# ============================================================

fig, ax = plt.subplots(
    3, 1,
    figsize=(10,8)
)

for i, fname in enumerate(files):

    sector = sector_labels[fname]

    time, flux = load_sector_file(fname)

    flat_flux, trend = detrend_flux(
        time,
        flux
    )

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB
    )

    phase = (tc % Prot) / Prot

    phase_plot = np.concatenate([
        phase,
        phase + 1
    ])

    flux_plot = np.concatenate([
        fc,
        fc
    ])

    ax[i].scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5
    )

    ax[i].set_xlim(0, 2)

    ax[i].set_ylabel("Flux")

    ax[i].set_title(
        f"{sector} | P_rot = {Prot:.3f} d"
    )

ax[-1].set_xlabel("Rotational Phase")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# TIC 427393298
# ORBITAL + ROTATIONAL ANALYSIS
# USING WOTAN DETRENDING
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import LombScargle
from wotan import flatten

# ============================================================
# CONFIG
# ============================================================

P_ORB = 3.9050

files = [
    "tic 427393298_s_6_corrected.csv",
    "tic_427393298_s_32_corrected.csv",
    "tic_427393298_s_98_corrected.csv"
]

sector_labels = {
    "tic 427393298_s_6_corrected.csv": "S6",
    "tic_427393298_s_32_corrected.csv": "S32",
    "tic_427393298_s_98_corrected.csv": "S98"
}

# ============================================================
# LOAD DATA
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(fname)

    print(f"\nColumns in {fname}:")
    print(df.columns)

    # --------------------------------------------------------
    # detect columns automatically
    # --------------------------------------------------------

    time_col = None
    flux_col = None

    for c in df.columns:

        cl = c.lower()

        if time_col is None and "time" in cl:
            time_col = c

        if flux_col is None and (
            "flux" in cl
            or "corr" in cl
            or "flat" in cl
        ):
            flux_col = c

    if time_col is None:
        raise ValueError("No TIME column found")

    if flux_col is None:
        raise ValueError("No FLUX column found")

    print(f"Using TIME column: {time_col}")
    print(f"Using FLUX column: {flux_col}")

    time = np.array(df[time_col])
    flux = np.array(df[flux_col])

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    return time, flux


# ============================================================
# WOTAN DETRENDING
# ============================================================

def detrend_flux(time,
                 flux,
                 P_orb,
                 eclipse_width=0.08,
                 window_length=0.5):

    time = np.asarray(time)
    flux = np.asarray(flux)

    # --------------------------------------------------------
    # clean NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # remove bad values
    # --------------------------------------------------------

    mask = flux > 0

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux = flux / np.nanmedian(flux)

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flux)]

    phase = ((time - t0) % P_orb) / P_orb - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # --------------------------------------------------------
    # WOTAN flatten
    # --------------------------------------------------------

    flat_flux, trend = flatten(
        time,
        flux,
        method='biweight',
        window_length=window_length,
        return_trend=True,
        mask=mask_eclipse
    )

    # --------------------------------------------------------
    # clean final NaNs
    # --------------------------------------------------------

    mask = np.isfinite(flat_flux)

    time = time[mask]
    flux = flux[mask]
    flat_flux = flat_flux[mask]
    trend = trend[mask]
    mask_eclipse = mask_eclipse[mask]

    return time, flux, flat_flux, trend, mask_eclipse


# ============================================================
# ROTATION PERIOD
# ============================================================

def get_rotation_period(time,
                        flux_flat,
                        P_orb,
                        eclipse_width=0.08,
                        min_period=1.0,
                        max_period=20):

    # --------------------------------------------------------
    # eclipse masking
    # --------------------------------------------------------

    t0 = time[np.argmin(flux_flat)]

    phase = ((time - t0) % P_orb) / P_orb - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    time_clean = time[~mask_eclipse]
    flux_clean = flux_flat[~mask_eclipse]

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time_clean) & np.isfinite(flux_clean)

    time_clean = time_clean[mask]
    flux_clean = flux_clean[mask]

    # --------------------------------------------------------
    # enough points?
    # --------------------------------------------------------

    if len(time_clean) < 10:
        raise ValueError("Not enough clean points")

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux_clean = flux_clean / np.median(flux_clean)

    # --------------------------------------------------------
    # Lomb-Scargle
    # --------------------------------------------------------

    freq, power = LombScargle(
        time_clean,
        flux_clean
    ).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period
    )

    periods = 1 / freq

    # --------------------------------------------------------
    # best period
    # --------------------------------------------------------

    Prot = periods[np.argmax(power)]

    # --------------------------------------------------------
    # check for 2P alias
    # --------------------------------------------------------

    if np.abs(Prot - P_orb/2) < 0.2:
        Prot = 2 * Prot
        print("⚠️ Using 2P alias correction")

    return Prot, periods, power, time_clean, flux_clean


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_sector(time,
                flux,
                flat_flux,
                trend,
                mask_eclipse,
                Prot,
                periods,
                power,
                sector):

    fig = plt.figure(figsize=(14,9))

    # ========================================================
    # RAW LIGHT CURVE
    # ========================================================

    ax1 = plt.subplot(221)

    ax1.plot(
        time,
        flux,
        '.k',
        ms=2,
        alpha=0.5,
        label='Raw Flux'
    )

    ax1.plot(
        time,
        trend,
        color='red',
        lw=2,
        label='WOTAN Trend'
    )

    ax1.set_title(f"{sector} — Raw + Trend")

    ax1.set_xlabel("Time [BTJD]")
    ax1.set_ylabel("Flux")

    ax1.legend()

    # ========================================================
    # DETRENDED
    # ========================================================

    ax2 = plt.subplot(222)

    ax2.plot(
        time,
        flat_flux,
        '.k',
        ms=2,
        alpha=0.5
    )

    ax2.scatter(
        time[mask_eclipse],
        flat_flux[mask_eclipse],
        color='red',
        s=5,
        label='Masked eclipses'
    )

    ax2.set_title(f"{sector} — WOTAN Detrended")

    ax2.set_xlabel("Time [BTJD]")
    ax2.set_ylabel("Normalized Flux")

    ax2.legend()

    # ========================================================
    # LOMB-SCARGLE
    # ========================================================

    ax3 = plt.subplot(223)

    ax3.plot(
        periods,
        power,
        color='k'
    )

    ax3.axvline(
        Prot,
        color='red',
        ls='--',
        lw=2,
        label=f"P_rot = {Prot:.3f} d"
    )

    ax3.axvline(
        P_ORB,
        color='C1',
        ls=':',
        lw=2,
        label=f"P_orb = {P_ORB:.3f} d"
    )

    ax3.set_xlim(0, 10)

    ax3.set_xlabel("Period [days]")
    ax3.set_ylabel("Power")

    ax3.set_title("Lomb-Scargle")

    ax3.legend()

    # ========================================================
    # ROTATIONAL PHASE
    # ========================================================

    ax4 = plt.subplot(224)

    phase_rot = (time % Prot) / Prot

    phase_plot = np.concatenate([
        phase_rot,
        phase_rot + 1
    ])

    flux_plot = np.concatenate([
        flat_flux,
        flat_flux
    ])

    ax4.scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5
    )

    ax4.set_xlim(0, 2)

    ax4.set_xlabel("Rotational Phase")
    ax4.set_ylabel("Normalized Flux")

    ax4.set_title(
        f"P_rot / P_orb = {Prot/P_ORB:.3f}"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# MAIN LOOP
# ============================================================

results = []

for fname in files:

    print("\n===================================================")
    print(f"PROCESSING: {fname}")

    sector = sector_labels[fname]

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    time, flux = load_sector_file(fname)

    # --------------------------------------------------------
    # DETREND
    # --------------------------------------------------------

    time, flux, flat_flux, trend, mask_eclipse = detrend_flux(
        time,
        flux,
        P_ORB,
        eclipse_width=0.08,
        window_length=0.5
    )

    # --------------------------------------------------------
    # ROTATION
    # --------------------------------------------------------

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB,
        min_period=1.0
    )

    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    results.append({
        "sector": sector,
        "P_rot": Prot,
        "P_orb": P_ORB,
        "ratio": Prot / P_ORB
    })

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print("\nRESULTS")
    print(f"P_orb = {P_ORB:.4f} d")
    print(f"P_rot = {Prot:.4f} d")
    print(f"P_rot / P_orb = {Prot/P_ORB:.3f}")

    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------

    plot_sector(
        time,
        flux,
        flat_flux,
        trend,
        mask_eclipse,
        Prot,
        periods,
        power,
        sector
    )

# ============================================================
# RESULTS TABLE
# ============================================================

results = pd.DataFrame(results)

print("\n===================================================")
print("FINAL RESULTS")
print(results)

# ============================================================
# EVOLUTION OF ROTATION
# ============================================================

plt.figure(figsize=(7,4))

sector_num = [
    int(s.replace("S",""))
    for s in results["sector"]
]

plt.plot(
    sector_num,
    results["P_rot"],
    '-o',
    lw=2
)

plt.axhline(
    P_ORB,
    color='red',
    ls='--',
    label='P_orb'
)

plt.xlabel("TESS Sector")
plt.ylabel("Rotation Period [days]")

plt.title("Evolution of Stellar Rotation")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

# ============================================================
# PHASE COMPARISON
# ============================================================

fig, ax = plt.subplots(
    3, 1,
    figsize=(10,8)
)

for i, fname in enumerate(files):

    sector = sector_labels[fname]

    time, flux = load_sector_file(fname)

    time, flux, flat_flux, trend, mask_eclipse = detrend_flux(
        time,
        flux,
        P_ORB
    )

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB
    )

    phase = (tc % Prot) / Prot

    phase_plot = np.concatenate([
        phase,
        phase + 1
    ])

    flux_plot = np.concatenate([
        fc,
        fc
    ])

    ax[i].scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5
    )

    ax[i].set_xlim(0, 2)

    ax[i].set_ylabel("Flux")

    ax[i].set_title(
        f"{sector} | P_rot = {Prot:.3f} d"
    )

ax[-1].set_xlabel("Rotational Phase")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TIC 427393298
# ORBITAL + ROTATIONAL ANALYSIS
# IMPROVED WOTAN VERSION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import LombScargle
from wotan import flatten

# ============================================================
# CONFIG
# ============================================================

P_ORB = 3.9050

files = [
    "tic 427393298_s_6_corrected.csv",
    "tic_427393298_s_32_corrected.csv",
    "tic_427393298_s_98_corrected.csv"
]

sector_labels = {
    "tic 427393298_s_6_corrected.csv": "S6",
    "tic_427393298_s_32_corrected.csv": "S32",
    "tic_427393298_s_98_corrected.csv": "S98"
}

# ============================================================
# LOAD DATA
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(fname)

    print(f"\nColumns in {fname}:")
    print(df.columns)

    # --------------------------------------------------------
    # detect columns automatically
    # --------------------------------------------------------

    time_col = None
    flux_col = None

    for c in df.columns:

        cl = c.lower()

        if time_col is None and "time" in cl:
            time_col = c

        if flux_col is None and (
            "flux" in cl
            or "corr" in cl
            or "flat" in cl
        ):
            flux_col = c

    if time_col is None:
        raise ValueError("No TIME column found")

    if flux_col is None:
        raise ValueError("No FLUX column found")

    print(f"Using TIME column: {time_col}")
    print(f"Using FLUX column: {flux_col}")

    time = np.array(df[time_col])
    flux = np.array(df[flux_col])

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    return time, flux


# ============================================================
# WOTAN DETRENDING
# ============================================================

def detrend_flux(time,
                 flux,
                 P_orb,
                 eclipse_width=0.06,
                 window_length=1.5):

    time = np.asarray(time)
    flux = np.asarray(flux)

    # --------------------------------------------------------
    # clean NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # remove bad values
    # --------------------------------------------------------

    mask = flux > 0

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux = flux / np.nanmedian(flux)

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flux)]

    phase = ((time - t0) % P_orb) / P_orb - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # --------------------------------------------------------
    # WOTAN flatten
    # --------------------------------------------------------

    flat_flux, trend = flatten(
        time,
        flux,
        method='biweight',
        window_length=window_length,
        return_trend=True,
        mask=mask_eclipse
    )

    # --------------------------------------------------------
    # clean final NaNs
    # --------------------------------------------------------

    mask = np.isfinite(flat_flux)

    time = time[mask]
    flux = flux[mask]
    flat_flux = flat_flux[mask]
    trend = trend[mask]
    mask_eclipse = mask_eclipse[mask]

    return time, flux, flat_flux, trend, mask_eclipse


# ============================================================
# ROTATION PERIOD
# ============================================================

def get_rotation_period(time,
                        flux_flat,
                        P_orb,
                        eclipse_width=0.06,
                        min_period=1.0,
                        max_period=20):

    # --------------------------------------------------------
    # eclipse masking
    # --------------------------------------------------------

    t0 = time[np.argmin(flux_flat)]

    phase = ((time - t0) % P_orb) / P_orb - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    time_clean = time[~mask_eclipse]
    flux_clean = flux_flat[~mask_eclipse]

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time_clean) & np.isfinite(flux_clean)

    time_clean = time_clean[mask]
    flux_clean = flux_clean[mask]

    # --------------------------------------------------------
    # enough points?
    # --------------------------------------------------------

    if len(time_clean) < 10:
        raise ValueError("Not enough clean points")

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux_clean = flux_clean / np.median(flux_clean)

    # --------------------------------------------------------
    # Lomb-Scargle
    # --------------------------------------------------------

    freq, power = LombScargle(
        time_clean,
        flux_clean
    ).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period,
        samples_per_peak=20
    )

    periods = 1 / freq

    # --------------------------------------------------------
    # best period
    # --------------------------------------------------------

    Prot = periods[np.argmax(power)]

    # --------------------------------------------------------
    # alias correction
    # --------------------------------------------------------

    if np.abs(Prot - P_orb/2) < 0.2:
        Prot = 2 * Prot
        print("⚠️ Using 2P alias correction")

    return Prot, periods, power, time_clean, flux_clean


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_sector(time,
                flux,
                flat_flux,
                trend,
                mask_eclipse,
                Prot,
                periods,
                power,
                tc,
                fc,
                sector):

    fig = plt.figure(figsize=(14,9))

    # ========================================================
    # RAW LIGHT CURVE
    # ========================================================

    ax1 = plt.subplot(221)

    ax1.plot(
        time,
        flux,
        '.k',
        ms=2,
        alpha=0.5,
        label='Raw Flux'
    )

    ax1.plot(
        time,
        trend,
        color='red',
        lw=2,
        label='WOTAN Trend'
    )

    ax1.set_title(f"{sector} — Raw + Trend")

    ax1.set_xlabel("Time [BTJD]")
    ax1.set_ylabel("Flux")

    ax1.legend()

    # ========================================================
    # DETRENDED
    # ========================================================

    ax2 = plt.subplot(222)

    ax2.plot(
        time,
        flat_flux,
        '.k',
        ms=2,
        alpha=0.5
    )

    ax2.scatter(
        time[mask_eclipse],
        flat_flux[mask_eclipse],
        color='red',
        s=5,
        label='Masked eclipses'
    )

    ax2.set_title(f"{sector} — WOTAN Detrended")

    ax2.set_xlabel("Time [BTJD]")
    ax2.set_ylabel("Normalized Flux")

    ax2.legend()

    # ========================================================
    # LOMB-SCARGLE
    # ========================================================

    ax3 = plt.subplot(223)

    ax3.plot(
        periods,
        power,
        color='k'
    )

    ax3.axvline(
        Prot,
        color='red',
        ls='--',
        lw=2,
        label=f"P_rot = {Prot:.3f} d"
    )

    ax3.axvline(
        P_ORB,
        color='orange',
        ls=':',
        lw=2,
        label=f"P_orb = {P_ORB:.3f} d"
    )

    ax3.set_xlim(0, 10)

    ax3.set_xlabel("Period [days]")
    ax3.set_ylabel("Power")

    ax3.set_title("Lomb-Scargle")

    ax3.legend()

    # ========================================================
    # ROTATIONAL PHASE (NO ECLIPSES)
    # ========================================================

    ax4 = plt.subplot(224)

    phase_rot = (tc % Prot) / Prot

    phase_plot = np.concatenate([
        phase_rot,
        phase_rot + 1
    ])

    flux_plot = np.concatenate([
        fc,
        fc
    ])

    ax4.scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5,
        color='C0'
    )

    ax4.set_xlim(0, 2)

    ax4.set_xlabel("Rotational Phase")
    ax4.set_ylabel("Normalized Flux")

    ax4.set_title(
        f"P_rot / P_orb = {Prot/P_ORB:.3f}"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# MAIN LOOP
# ============================================================

results = []

for fname in files:

    print("\n===================================================")
    print(f"PROCESSING: {fname}")

    sector = sector_labels[fname]

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    time, flux = load_sector_file(fname)

    # --------------------------------------------------------
    # DETREND
    # --------------------------------------------------------

    time, flux, flat_flux, trend, mask_eclipse = detrend_flux(
        time,
        flux,
        P_ORB,
        eclipse_width=0.06,
        window_length=1.5
    )

    # --------------------------------------------------------
    # ROTATION
    # --------------------------------------------------------

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB,
        eclipse_width=0.06,
        min_period=1.0
    )

    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    results.append({
        "sector": sector,
        "P_rot": Prot,
        "P_orb": P_ORB,
        "ratio": Prot / P_ORB
    })

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print("\nRESULTS")
    print(f"P_orb = {P_ORB:.4f} d")
    print(f"P_rot = {Prot:.4f} d")
    print(f"P_rot / P_orb = {Prot/P_ORB:.3f}")

    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------

    plot_sector(
        time,
        flux,
        flat_flux,
        trend,
        mask_eclipse,
        Prot,
        periods,
        power,
        tc,
        fc,
        sector
    )

# ============================================================
# RESULTS TABLE
# ============================================================

results = pd.DataFrame(results)

print("\n===================================================")
print("FINAL RESULTS")
print(results)

# ============================================================
# EVOLUTION OF ROTATION
# ============================================================

plt.figure(figsize=(7,4))

sector_num = [
    int(s.replace("S",""))
    for s in results["sector"]
]

plt.plot(
    sector_num,
    results["P_rot"],
    '-o',
    lw=2
)

plt.axhline(
    P_ORB,
    color='red',
    ls='--',
    label='P_orb'
)

plt.xlabel("TESS Sector")
plt.ylabel("Rotation Period [days]")

plt.title("Evolution of Stellar Rotation")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

# ============================================================
# PHASE COMPARISON
# ============================================================

fig, ax = plt.subplots(
    3, 1,
    figsize=(10,8)
)

for i, fname in enumerate(files):

    sector = sector_labels[fname]

    time, flux = load_sector_file(fname)

    time, flux, flat_flux, trend, mask_eclipse = detrend_flux(
        time,
        flux,
        P_ORB
    )

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB
    )

    phase = (tc % Prot) / Prot

    phase_plot = np.concatenate([
        phase,
        phase + 1
    ])

    flux_plot = np.concatenate([
        fc,
        fc
    ])

    ax[i].scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5
    )

    ax[i].set_xlim(0, 2)

    ax[i].set_ylabel("Flux")

    ax[i].set_title(
        f"{sector} | P_rot = {Prot:.3f} d"
    )

ax[-1].set_xlabel("Rotational Phase")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TIC 427393298
# ORBITAL + ROTATIONAL ANALYSIS
# FINAL IMPROVED PIPELINE
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import LombScargle
from wotan import flatten

# ============================================================
# CONFIG
# ============================================================

P_ORB = 3.9050

files = [
    "tic 427393298_s_6_corrected.csv",
    "tic_427393298_s_32_corrected.csv",
    "tic_427393298_s_98_corrected.csv"
]

sector_labels = {
    "tic 427393298_s_6_corrected.csv": "S6",
    "tic_427393298_s_32_corrected.csv": "S32",
    "tic_427393298_s_98_corrected.csv": "S98"
}

# ============================================================
# LOAD DATA
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(fname)

    print(f"\nColumns in {fname}:")
    print(df.columns)

    # --------------------------------------------------------
    # detect columns automatically
    # --------------------------------------------------------

    time_col = None
    flux_col = None

    for c in df.columns:

        cl = c.lower()

        if time_col is None and "time" in cl:
            time_col = c

        if flux_col is None and (
            "flux" in cl
            or "corr" in cl
            or "flat" in cl
        ):
            flux_col = c

    if time_col is None:
        raise ValueError("No TIME column found")

    if flux_col is None:
        raise ValueError("No FLUX column found")

    print(f"Using TIME column: {time_col}")
    print(f"Using FLUX column: {flux_col}")

    time = np.array(df[time_col])
    flux = np.array(df[flux_col])

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    return time, flux


# ============================================================
# IMPROVED WOTAN DETRENDING
# ============================================================

def detrend_flux(time,
                 flux,
                 P_orb,
                 sector,
                 eclipse_width=0.06):

    time = np.asarray(time)
    flux = np.asarray(flux)

    # --------------------------------------------------------
    # clean NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # remove bad values
    # --------------------------------------------------------

    mask = flux > 0

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux = flux / np.nanmedian(flux)

    # --------------------------------------------------------
    # dynamic window length
    # --------------------------------------------------------

    if sector in ["S6", "S32"]:
        window_length = 0.75
    else:
        window_length = 1.5

    print(f"Using window_length = {window_length}")

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flux)]

    phase = ((time - t0) % P_orb) / P_orb - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # --------------------------------------------------------
    # WOTAN detrending
    # --------------------------------------------------------

    flat_flux, trend = flatten(
        time,
        flux,
        method='lowess',
        window_length=window_length,
        return_trend=True,
        mask=mask_eclipse
    )

    # --------------------------------------------------------
    # remove NaNs after detrending
    # --------------------------------------------------------

    mask = np.isfinite(flat_flux)

    time = time[mask]
    flux = flux[mask]
    flat_flux = flat_flux[mask]
    trend = trend[mask]
    mask_eclipse = mask_eclipse[mask]

    return time, flux, flat_flux, trend, mask_eclipse


# ============================================================
# ROTATION PERIOD
# ============================================================

def get_rotation_period(time,
                        flux_flat,
                        P_orb,
                        eclipse_width=0.06,
                        min_period=1.0,
                        max_period=20):

    # --------------------------------------------------------
    # eclipse masking
    # --------------------------------------------------------

    t0 = time[np.argmin(flux_flat)]

    phase = ((time - t0) % P_orb) / P_orb - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    time_clean = time[~mask_eclipse]
    flux_clean = flux_flat[~mask_eclipse]

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time_clean) & np.isfinite(flux_clean)

    time_clean = time_clean[mask]
    flux_clean = flux_clean[mask]

    # --------------------------------------------------------
    # enough points?
    # --------------------------------------------------------

    if len(time_clean) < 10:
        raise ValueError("Not enough clean points")

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux_clean = flux_clean / np.median(flux_clean)

    # --------------------------------------------------------
    # Lomb-Scargle
    # --------------------------------------------------------

    freq, power = LombScargle(
        time_clean,
        flux_clean
    ).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period,
        samples_per_peak=20
    )

    periods = 1 / freq

    # --------------------------------------------------------
    # best period
    # --------------------------------------------------------

    Prot = periods[np.argmax(power)]

    # --------------------------------------------------------
    # alias correction
    # --------------------------------------------------------

    if np.abs(Prot - P_orb/2) < 0.2:
        Prot = 2 * Prot
        print("⚠️ Using 2P alias correction")

    return Prot, periods, power, time_clean, flux_clean


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_sector(time,
                flux,
                flat_flux,
                trend,
                mask_eclipse,
                Prot,
                periods,
                power,
                tc,
                fc,
                sector):

    fig = plt.figure(figsize=(14,9))

    # ========================================================
    # RAW LIGHT CURVE
    # ========================================================

    ax1 = plt.subplot(221)

    ax1.plot(
        time,
        flux,
        '.k',
        ms=2,
        alpha=0.5,
        label='Raw Flux'
    )

    ax1.plot(
        time,
        trend,
        color='red',
        lw=2,
        label='WOTAN Trend'
    )

    ax1.set_title(f"{sector} — Raw + Trend")

    ax1.set_xlabel("Time [BTJD]")
    ax1.set_ylabel("Flux")

    ax1.legend()

    # ========================================================
    # DETRENDED
    # ========================================================

    ax2 = plt.subplot(222)

    ax2.plot(
        time,
        flat_flux,
        '.k',
        ms=2,
        alpha=0.5
    )

    ax2.scatter(
        time[mask_eclipse],
        flat_flux[mask_eclipse],
        color='red',
        s=5,
        label='Masked eclipses'
    )

    ax2.set_title(f"{sector} — WOTAN Detrended")

    ax2.set_xlabel("Time [BTJD]")
    ax2.set_ylabel("Normalized Flux")

    ax2.legend()

    # ========================================================
    # LOMB-SCARGLE
    # ========================================================

    ax3 = plt.subplot(223)

    ax3.plot(
        periods,
        power,
        color='k'
    )

    ax3.axvline(
        Prot,
        color='red',
        ls='--',
        lw=2,
        label=f"P_rot = {Prot:.3f} d"
    )

    ax3.axvline(
        P_ORB,
        color='orange',
        ls=':',
        lw=2,
        label=f"P_orb = {P_ORB:.3f} d"
    )

    ax3.set_xlim(0, 10)

    ax3.set_xlabel("Period [days]")
    ax3.set_ylabel("Power")

    ax3.set_title("Lomb-Scargle")

    ax3.legend()

    # ========================================================
    # ROTATIONAL PHASE (NO ECLIPSES)
    # ========================================================

    ax4 = plt.subplot(224)

    phase_rot = (tc % Prot) / Prot

    phase_plot = np.concatenate([
        phase_rot,
        phase_rot + 1
    ])

    flux_plot = np.concatenate([
        fc,
        fc
    ])

    ax4.scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5,
        color='C0'
    )

    ax4.set_xlim(0, 2)

    ax4.set_xlabel("Rotational Phase")
    ax4.set_ylabel("Normalized Flux")

    ax4.set_title(
        f"P_rot / P_orb = {Prot/P_ORB:.3f}"
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# MAIN LOOP
# ============================================================

results = []

for fname in files:

    print("\n===================================================")
    print(f"PROCESSING: {fname}")

    sector = sector_labels[fname]

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    time, flux = load_sector_file(fname)

    # --------------------------------------------------------
    # DETREND
    # --------------------------------------------------------

    time, flux, flat_flux, trend, mask_eclipse = detrend_flux(
        time,
        flux,
        P_ORB,
        sector
    )

    # --------------------------------------------------------
    # ROTATION
    # --------------------------------------------------------

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB,
        eclipse_width=0.06,
        min_period=1.0
    )

    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    results.append({
        "sector": sector,
        "P_rot": Prot,
        "P_orb": P_ORB,
        "ratio": Prot / P_ORB
    })

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print("\nRESULTS")
    print(f"P_orb = {P_ORB:.4f} d")
    print(f"P_rot = {Prot:.4f} d")
    print(f"P_rot / P_orb = {Prot/P_ORB:.3f}")

    # --------------------------------------------------------
    # PLOTS
    # --------------------------------------------------------

    plot_sector(
        time,
        flux,
        flat_flux,
        trend,
        mask_eclipse,
        Prot,
        periods,
        power,
        tc,
        fc,
        sector
    )

# ============================================================
# RESULTS TABLE
# ============================================================

results = pd.DataFrame(results)

print("\n===================================================")
print("FINAL RESULTS")
print(results)

# ============================================================
# EVOLUTION OF ROTATION
# ============================================================

plt.figure(figsize=(7,4))

sector_num = [
    int(s.replace("S",""))
    for s in results["sector"]
]

plt.plot(
    sector_num,
    results["P_rot"],
    '-o',
    lw=2
)

plt.axhline(
    P_ORB,
    color='red',
    ls='--',
    label='P_orb'
)

plt.xlabel("TESS Sector")
plt.ylabel("Rotation Period [days]")

plt.title("Evolution of Stellar Rotation")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

# ============================================================
# PHASE COMPARISON
# ============================================================

fig, ax = plt.subplots(
    3, 1,
    figsize=(10,8)
)

for i, fname in enumerate(files):

    sector = sector_labels[fname]

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    time, flux = load_sector_file(fname)

    # --------------------------------------------------------
    # DETREND
    # --------------------------------------------------------

    time, flux, flat_flux, trend, mask_eclipse = detrend_flux(
        time,
        flux,
        P_ORB,
        sector
    )

    # --------------------------------------------------------
    # ROTATION
    # --------------------------------------------------------

    Prot, periods, power, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_ORB
    )

    # --------------------------------------------------------
    # PHASE FOLD
    # --------------------------------------------------------

    phase = (tc % Prot) / Prot

    phase_plot = np.concatenate([
        phase,
        phase + 1
    ])

    flux_plot = np.concatenate([
        fc,
        fc
    ])

    ax[i].scatter(
        phase_plot,
        flux_plot,
        s=4,
        alpha=0.5
    )

    ax[i].set_xlim(0, 2)

    ax[i].set_ylabel("Flux")

    ax[i].set_title(
        f"{sector} | P_rot = {Prot:.3f} d"
    )

ax[-1].set_xlabel("Rotational Phase")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MULTI-TIC ORBITAL + ROTATIONAL ANALYSIS
# FINAL AUTO-BLS VERSION
# ============================================================

import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import (
    LombScargle,
    BoxLeastSquares
)

from wotan import flatten

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = (
    "/home/luciam/Documents/starry/"
    "Orion_yceb/dat_orion_sergio/sample_3/"
    "831359(1)/Light_Curve_Files/"
)

OUTPUT_DIR = "rotation_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# FIND FILES
# ============================================================

files = sorted(
    glob.glob(os.path.join(BASE_PATH, "*.txt"))
)

print(f"\nFound {len(files)} files")

# ============================================================
# GROUP FILES BY TIC
# ============================================================

tic_files = {}

for f in files:

    fname = os.path.basename(f)

    match = re.search(
        r'tic\s+(\d+)_S_(\d+)_LC',
        fname
    )

    if match is None:
        continue

    tic = int(match.group(1))
    sector = int(match.group(2))

    if tic not in tic_files:
        tic_files[tic] = []

    tic_files[tic].append((sector, f))

# ============================================================
# LOAD LIGHT CURVE
# ============================================================

def load_sector_file(fname):

    # --------------------------------------------------------
    # load
    # --------------------------------------------------------

    try:

        df = pd.read_csv(
            fname,
            delim_whitespace=True,
            comment='#',
            header=None
        )

    except Exception as e:

        raise ValueError(
            f"Could not load {fname}: {e}"
        )

    arr = df.values

    ncols = arr.shape[1]

    print(f"\n{os.path.basename(fname)}")
    print(f"Detected {ncols} columns")

    # --------------------------------------------------------
    # detect TIME column
    # --------------------------------------------------------

    time_col = None

    for i in range(ncols):

        col = arr[:,i]

        med = np.nanmedian(col)

        # BTJD range
        if 1000 < med < 5000:

            time_col = i
            break

    if time_col is None:
        raise ValueError(
            "Could not identify TIME column"
        )

    # --------------------------------------------------------
    # detect FLUX column
    # --------------------------------------------------------

    flux_col = None

    for i in range(ncols):

        if i == time_col:
            continue

        col = arr[:,i]

        med = np.nanmedian(col)

        # normalized flux
        if 0.5 < med < 1.5:

            flux_col = i
            break

    if flux_col is None:
        raise ValueError(
            "Could not identify FLUX column"
        )

    # --------------------------------------------------------
    # extract
    # --------------------------------------------------------

    time = arr[:,time_col]
    flux = arr[:,flux_col]

    print(f"TIME column = {time_col}")
    print(f"FLUX column = {flux_col}")

    # --------------------------------------------------------
    # clean NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # sort by time
    # --------------------------------------------------------

    srt = np.argsort(time)

    time = time[srt]
    flux = flux[srt]

    return time, flux


# ============================================================
# FIND ORBITAL PERIOD WITH BLS
# ============================================================

def get_orbital_period(time,
                       flux,
                       min_period=0.5,
                       max_period=10):

    flux = flux / np.nanmedian(flux)

    # --------------------------------------------------------
    # BLS durations
    # --------------------------------------------------------

    durations = np.linspace(
        0.02,
        0.2,
        20
    )

    # --------------------------------------------------------
    # BLS
    # --------------------------------------------------------

    bls = BoxLeastSquares(
        time,
        flux
    )

    periods = np.linspace(
        min_period,
        max_period,
        10000
    )

    result = bls.power(
        periods,
        durations
    )

    # --------------------------------------------------------
    # best solution
    # --------------------------------------------------------

    best_idx = np.argmax(result.power)

    P_orb = result.period[best_idx]

    t0 = result.transit_time[best_idx]

    depth = result.depth[best_idx]

    return P_orb, t0, depth, result


# ============================================================
# DETRENDING
# ============================================================

def detrend_flux(time,
                 flux,
                 P_orb,
                 sector,
                 eclipse_width=0.06):

    time = np.asarray(time)
    flux = np.asarray(flux)

    # --------------------------------------------------------
    # clean NaNs
    # --------------------------------------------------------

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # positive flux only
    # --------------------------------------------------------

    mask = flux > 0

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux = flux / np.nanmedian(flux)

    # --------------------------------------------------------
    # dynamic window
    # --------------------------------------------------------

    if sector in [5, 6, 32, 33]:

        window_length = 0.75

    else:

        window_length = 1.5

    print(
        f"window_length = {window_length}"
    )

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flux)]

    phase = (
        ((time - t0) % P_orb) / P_orb
    ) - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # --------------------------------------------------------
    # WOTAN
    # --------------------------------------------------------

    flat_flux, trend = flatten(
        time,
        flux,
        method='lowess',
        window_length=window_length,
        return_trend=True,
        mask=mask_eclipse
    )

    # --------------------------------------------------------
    # clean NaNs
    # --------------------------------------------------------

    mask = np.isfinite(flat_flux)

    time = time[mask]
    flux = flux[mask]
    flat_flux = flat_flux[mask]
    trend = trend[mask]
    mask_eclipse = mask_eclipse[mask]

    return (
        time,
        flux,
        flat_flux,
        trend,
        mask_eclipse
    )


# ============================================================
# ROTATION PERIOD
# ============================================================

def get_rotation_period(time,
                        flux_flat,
                        P_orb,
                        eclipse_width=0.06,
                        min_period=0.5,
                        max_period=20):

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flux_flat)]

    phase = (
        ((time - t0) % P_orb) / P_orb
    ) - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width) |
        (np.abs(phase - 0.5) < eclipse_width) |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    time_clean = time[~mask_eclipse]
    flux_clean = flux_flat[~mask_eclipse]

    # --------------------------------------------------------
    # remove NaNs
    # --------------------------------------------------------

    mask = (
        np.isfinite(time_clean)
        &
        np.isfinite(flux_clean)
    )

    time_clean = time_clean[mask]
    flux_clean = flux_clean[mask]

    if len(time_clean) < 20:
        raise ValueError(
            "Not enough clean points"
        )

    # --------------------------------------------------------
    # normalize
    # --------------------------------------------------------

    flux_clean = (
        flux_clean /
        np.median(flux_clean)
    )

    # --------------------------------------------------------
    # Lomb-Scargle
    # --------------------------------------------------------

    freq, power = LombScargle(
        time_clean,
        flux_clean
    ).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period,
        samples_per_peak=20
    )

    periods = 1 / freq

    Prot = periods[np.argmax(power)]

    return (
        Prot,
        periods,
        power,
        time_clean,
        flux_clean
    )


# ============================================================
# MAIN LOOP
# ============================================================

summary_results = []

for tic in sorted(tic_files.keys()):

    print("\n===================================================")
    print(f"TIC {tic}")

    # --------------------------------------------------------
    # SORT SECTORS
    # --------------------------------------------------------

    sector_data = sorted(
        tic_files[tic]
    )

    # --------------------------------------------------------
    # FIRST SECTOR FOR BLS
    # --------------------------------------------------------

    sector0, file0 = sector_data[0]

    time0, flux0 = load_sector_file(file0)

    # --------------------------------------------------------
    # DETECT ORBITAL PERIOD
    # --------------------------------------------------------

    P_orb, t0, depth, bls_result = (
        get_orbital_period(
            time0,
            flux0
        )
    )

    print(f"\nDetected P_orb = {P_orb:.5f} d")
    print(f"Depth = {depth:.5f}")

    # --------------------------------------------------------
    # CREATE FIGURE
    # --------------------------------------------------------

    nsec = len(sector_data)

    fig, axes = plt.subplots(
        nsec,
        4,
        figsize=(18, 4*nsec)
    )

    if nsec == 1:
        axes = np.array([axes])

    # --------------------------------------------------------
    # LOOP SECTORS
    # --------------------------------------------------------

    for i, (sector, fname) in enumerate(sector_data):

        print(f"\nSector {sector}")

        try:

            # =================================================
            # LOAD
            # =================================================

            time, flux = load_sector_file(fname)

            # =================================================
            # DETREND
            # =================================================

            (
                time,
                flux,
                flat_flux,
                trend,
                mask_eclipse
            ) = detrend_flux(
                time,
                flux,
                P_orb,
                sector
            )

            # =================================================
            # ROTATION
            # =================================================

            (
                Prot,
                periods,
                power,
                tc,
                fc
            ) = get_rotation_period(
                time,
                flat_flux,
                P_orb
            )

            ratio = Prot / P_orb

            print(f"P_rot = {Prot:.5f}")
            print(f"P_rot/P_orb = {ratio:.3f}")

            # =================================================
            # SAVE SUMMARY
            # =================================================

            summary_results.append({
                "tic": tic,
                "sector": sector,
                "P_orb": P_orb,
                "P_rot": Prot,
                "ratio": ratio
            })

            # =================================================
            # AXES
            # =================================================

            ax1, ax2, ax3, ax4 = axes[i]

            # =================================================
            # RAW + TREND
            # =================================================

            ax1.plot(
                time,
                flux,
                '.k',
                ms=2,
                alpha=0.5
            )

            ax1.plot(
                time,
                trend,
                color='red',
                lw=2
            )

            ax1.set_title(
                f"TIC {tic} | S{sector}\nRaw + Trend"
            )

            ax1.set_xlabel("Time")
            ax1.set_ylabel("Flux")

            # =================================================
            # DETRENDED
            # =================================================

            ax2.plot(
                time,
                flat_flux,
                '.k',
                ms=2,
                alpha=0.5
            )

            ax2.scatter(
                time[mask_eclipse],
                flat_flux[mask_eclipse],
                color='red',
                s=5
            )

            ax2.set_title("Detrended")

            ax2.set_xlabel("Time")

            # =================================================
            # LOMB-SCARGLE
            # =================================================

            ax3.plot(
                periods,
                power,
                color='k'
            )

            ax3.axvline(
                Prot,
                color='red',
                ls='--',
                label=f"P_rot={Prot:.3f}"
            )

            ax3.axvline(
                P_orb,
                color='orange',
                ls=':',
                label=f"P_orb={P_orb:.3f}"
            )

            ax3.set_xlim(0, 10)

            ax3.set_title("Lomb-Scargle")

            ax3.set_xlabel("Period [d]")
            ax3.set_ylabel("Power")

            ax3.legend(fontsize=8)

            # =================================================
            # ROTATIONAL PHASE
            # =================================================

            phase = (
                (tc % Prot) / Prot
            )

            phase_plot = np.concatenate([
                phase,
                phase + 1
            ])

            flux_plot = np.concatenate([
                fc,
                fc
            ])

            ax4.scatter(
                phase_plot,
                flux_plot,
                s=4,
                alpha=0.5
            )

            ax4.set_xlim(0, 2)

            ax4.set_title(
                f"P_rot/P_orb = {ratio:.3f}"
            )

            ax4.set_xlabel(
                "Rotational Phase"
            )

        except Exception as e:

            print(f"ERROR: {e}")

    plt.tight_layout()

    # =========================================================
    # SAVE FIGURE
    # =========================================================

    outname = os.path.join(
        OUTPUT_DIR,
        f"TIC_{tic}_rotation_analysis.png"
    )

    plt.savefig(
        outname,
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    print(f"\nSaved: {outname}")

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_results = pd.DataFrame(
    summary_results
)

summary_csv = os.path.join(
    OUTPUT_DIR,
    "rotation_summary.csv"
)

summary_results.to_csv(
    summary_csv,
    index=False
)

print("\n===================================================")
print("FINAL SUMMARY")
print(summary_results)

print(f"\nSaved:")
print(summary_csv)

# ============================================================
# GLOBAL DISTRIBUTION
# ============================================================

plt.figure(figsize=(7,5))

plt.hist(
    summary_results["ratio"],
    bins=20
)

plt.axvline(
    1.0,
    color='red',
    ls='--',
    label='Synchronized'
)

plt.xlabel("P_rot / P_orb")
plt.ylabel("Number of sectors")

plt.title(
    "Synchronization Distribution"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "synchronization_distribution.png"
    ),
    dpi=300
)

plt.show()

print("\nDONE.")

In [ ]:
# ============================================================
# ORION YCEB
# FULL SCIENCE PIPELINE
# ============================================================
#
# This pipeline:
#
# 1. Reads all TESS light curves
# 2. Detects orbital periods with BLS
# 3. Detrends using WOTAN
# 4. Computes rotational periods
# 5. Identifies synchronized systems
# 6. Rejects contaminants
# 7. Produces publication-quality plots
#
# ============================================================

import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import (
    LombScargle,
    BoxLeastSquares
)

from wotan import flatten

# ============================================================
# CONFIG
# ============================================================

BASE_PATH = (
    "/home/luciam/Documents/starry/"
    "Orion_yceb/dat_orion_sergio/sample_3/"
    "831359(1)/Light_Curve_Files/"
)

OUTPUT_DIR = "science_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# FIND FILES
# ============================================================

files = sorted(
    glob.glob(os.path.join(BASE_PATH, "*.txt"))
)

print(f"\nFound {len(files)} files")

# ============================================================
# GROUP FILES BY TIC
# ============================================================

tic_files = {}

for f in files:

    fname = os.path.basename(f)

    match = re.search(
        r'tic\s+(\d+)_S_(\d+)_LC',
        fname
    )

    if match is None:
        continue

    tic = int(match.group(1))
    sector = int(match.group(2))

    if tic not in tic_files:
        tic_files[tic] = []

    tic_files[tic].append((sector, f))

# ============================================================
# LOAD LIGHT CURVE
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(
        fname,
        delim_whitespace=True,
        comment='#',
        header=None
    )

    arr = df.values

    ncols = arr.shape[1]

    # --------------------------------------------------------
    # detect time column
    # --------------------------------------------------------

    time_col = None

    for i in range(ncols):

        med = np.nanmedian(arr[:,i])

        if 1000 < med < 5000:
            time_col = i
            break

    # --------------------------------------------------------
    # detect flux column
    # --------------------------------------------------------

    flux_col = None

    for i in range(ncols):

        if i == time_col:
            continue

        med = np.nanmedian(arr[:,i])

        if 0.5 < med < 1.5:
            flux_col = i
            break

    if time_col is None:
        raise ValueError("TIME column not found")

    if flux_col is None:
        raise ValueError("FLUX column not found")

    time = arr[:,time_col]
    flux = arr[:,flux_col]

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    srt = np.argsort(time)

    time = time[srt]
    flux = flux[srt]

    return time, flux

# ============================================================
# BLS ORBITAL PERIOD
# ============================================================

def get_orbital_period(time,
                       flux,
                       min_period=0.3,
                       max_period=10):

    flux = flux / np.nanmedian(flux)

    durations = np.linspace(
        0.02,
        0.2,
        20
    )

    bls = BoxLeastSquares(
        time,
        flux
    )

    periods = np.linspace(
        min_period,
        max_period,
        15000
    )

    result = bls.power(
        periods,
        durations
    )

    best = np.argmax(result.power)

    P_orb = result.period[best]

    t0 = result.transit_time[best]

    depth = result.depth[best]

    return P_orb, t0, depth, result

# ============================================================
# DETREND
# ============================================================

def detrend_flux(time,
                 flux,
                 P_orb,
                 sector,
                 eclipse_width=0.06):

    flux = flux / np.nanmedian(flux)

    # --------------------------------------------------------
    # dynamic window
    # --------------------------------------------------------

    if sector in [5, 6, 32, 33]:
        window_length = 0.75
    else:
        window_length = 1.5

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flux)]

    phase = (
        ((time - t0) % P_orb) / P_orb
    ) - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width)
        |
        (np.abs(phase - 0.5) < eclipse_width)
        |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # --------------------------------------------------------
    # WOTAN
    # --------------------------------------------------------

    flat_flux, trend = flatten(
        time,
        flux,
        method='lowess',
        window_length=window_length,
        return_trend=True,
        mask=mask_eclipse
    )

    mask = np.isfinite(flat_flux)

    return (
        time[mask],
        flux[mask],
        flat_flux[mask],
        trend[mask],
        mask_eclipse[mask]
    )

# ============================================================
# ROTATION PERIOD
# ============================================================

def get_rotation_period(time,
                        flat_flux,
                        P_orb,
                        eclipse_width=0.06,
                        min_period=0.3,
                        max_period=20):

    t0 = time[np.argmin(flat_flux)]

    phase = (
        ((time - t0) % P_orb) / P_orb
    ) - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width)
        |
        (np.abs(phase - 0.5) < eclipse_width)
        |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    time_clean = time[~mask_eclipse]
    flux_clean = flat_flux[~mask_eclipse]

    flux_clean /= np.nanmedian(flux_clean)

    freq, power = LombScargle(
        time_clean,
        flux_clean
    ).autopower(
        minimum_frequency=1/max_period,
        maximum_frequency=1/min_period,
        samples_per_peak=20
    )

    periods = 1 / freq

    Prot = periods[np.argmax(power)]

    return (
        Prot,
        periods,
        power,
        time_clean,
        flux_clean
    )

# ============================================================
# HARMONIC TEST
# ============================================================

def harmonic_check(Prot,
                   P_orb):

    ratio = Prot / P_orb

    # synchronized
    if 0.9 < ratio < 1.1:
        return "synchronized"

    # harmonic
    elif 0.45 < ratio < 0.55:
        return "harmonic"

    # asynchronous
    else:
        return "asynchronous"

# ============================================================
# MAIN LOOP
# ============================================================

summary = []

for tic in sorted(tic_files.keys()):

    print("\n===================================================")
    print(f"TIC {tic}")

    sectors = sorted(tic_files[tic])

    # --------------------------------------------------------
    # orbital period from first sector
    # --------------------------------------------------------

    sector0, file0 = sectors[0]

    time0, flux0 = load_sector_file(file0)

    P_orb, t0, depth, bls_result = get_orbital_period(
        time0,
        flux0
    )

    print(f"P_orb = {P_orb:.5f} d")

    # ========================================================
    # FIGURE
    # ========================================================

    nsec = len(sectors)

    fig, axes = plt.subplots(
        nsec,
        4,
        figsize=(18, 4*nsec)
    )

    if nsec == 1:
        axes = np.array([axes])

    # ========================================================
    # LOOP SECTORS
    # ========================================================

    for i, (sector, fname) in enumerate(sectors):

        try:

            print(f"Sector {sector}")

            # ------------------------------------------------
            # LOAD
            # ------------------------------------------------

            time, flux = load_sector_file(fname)

            # ------------------------------------------------
            # DETREND
            # ------------------------------------------------

            (
                time,
                flux,
                flat_flux,
                trend,
                mask_eclipse
            ) = detrend_flux(
                time,
                flux,
                P_orb,
                sector
            )

            # ------------------------------------------------
            # ROTATION
            # ------------------------------------------------

            (
                Prot,
                periods,
                power,
                tc,
                fc
            ) = get_rotation_period(
                time,
                flat_flux,
                P_orb
            )

            ratio = Prot / P_orb

            classification = harmonic_check(
                Prot,
                P_orb
            )

            print(f"P_rot = {Prot:.5f}")
            print(f"ratio = {ratio:.3f}")
            print(f"class = {classification}")

            # ------------------------------------------------
            # SAVE
            # ------------------------------------------------

            summary.append({

                "tic": tic,
                "sector": sector,

                "P_orb": P_orb,
                "P_rot": Prot,

                "ratio": ratio,

                "classification": classification

            })

            # =================================================
            # PLOTS
            # =================================================

            ax1, ax2, ax3, ax4 = axes[i]

            # ------------------------------------------------
            # RAW
            # ------------------------------------------------

            ax1.plot(
                time,
                flux,
                '.k',
                ms=2,
                alpha=0.5
            )

            ax1.plot(
                time,
                trend,
                color='red',
                lw=2
            )

            ax1.set_title(
                f"TIC {tic} | S{sector}"
            )

            ax1.set_xlabel("Time")
            ax1.set_ylabel("Flux")

            # ------------------------------------------------
            # DETRENDED
            # ------------------------------------------------

            ax2.plot(
                time,
                flat_flux,
                '.k',
                ms=2,
                alpha=0.5
            )

            ax2.scatter(
                time[mask_eclipse],
                flat_flux[mask_eclipse],
                color='red',
                s=5
            )

            ax2.set_title("Detrended")

            # ------------------------------------------------
            # LS
            # ------------------------------------------------

            ax3.plot(
                periods,
                power,
                color='k'
            )

            ax3.axvline(
                Prot,
                color='red',
                ls='--',
                label=f"P_rot={Prot:.3f}"
            )

            ax3.axvline(
                P_orb,
                color='orange',
                ls=':',
                label=f"P_orb={P_orb:.3f}"
            )

            ax3.set_xlim(0, 10)

            ax3.legend(fontsize=8)

            ax3.set_title("Lomb-Scargle")

            ax3.set_xlabel("Period [d]")

            # ------------------------------------------------
            # PHASE
            # ------------------------------------------------

            phase = (tc % Prot) / Prot

            phase_plot = np.concatenate([
                phase,
                phase + 1
            ])

            flux_plot = np.concatenate([
                fc,
                fc
            ])

            ax4.scatter(
                phase_plot,
                flux_plot,
                s=4,
                alpha=0.5
            )

            ax4.set_xlim(0, 2)

            ax4.set_title(
                f"{classification}\n"
                f"ratio={ratio:.3f}"
            )

            ax4.set_xlabel(
                "Rotational Phase"
            )

        except Exception as e:

            print(f"ERROR: {e}")

    plt.tight_layout()

    outname = os.path.join(
        OUTPUT_DIR,
        f"TIC_{tic}.png"
    )

    plt.savefig(
        outname,
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

# ============================================================
# FINAL TABLE
# ============================================================

summary = pd.DataFrame(summary)

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "rotation_summary.csv"
    ),
    index=False
)

print("\n===================================================")
print(summary)

# ============================================================
# CLEAN SAMPLE
# ============================================================

clean = summary[
    (
        (summary["ratio"] > 0.8)
        &
        (summary["ratio"] < 1.2)
    )
]

print("\n===================================================")
print("LIKELY SYNCHRONIZED SYSTEMS")
print(clean)

# ============================================================
# HISTOGRAM
# ============================================================

plt.figure(figsize=(7,5))

plt.hist(
    summary["ratio"],
    bins=20
)

plt.axvline(
    1.0,
    color='red',
    ls='--',
    label='Synchronization'
)

plt.axvline(
    0.5,
    color='orange',
    ls=':',
    label='Harmonic'
)

plt.xlabel("P_rot / P_orb")
plt.ylabel("N")

plt.legend()

plt.title(
    "Synchronization Distribution"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "ratio_histogram.png"
    ),
    dpi=300
)

plt.show()

# ============================================================
# P_rot vs P_orb
# ============================================================

plt.figure(figsize=(6,6))

plt.scatter(
    summary["P_orb"],
    summary["P_rot"]
)

x = np.linspace(
    0,
    10,
    100
)

plt.plot(
    x,
    x,
    '--r',
    label='1:1'
)

plt.xlabel("P_orb [d]")
plt.ylabel("P_rot [d]")

plt.legend()

plt.title(
    "Rotational Synchronization"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "Prot_vs_Porb.png"
    ),
    dpi=300
)

plt.show()

print("\nDONE.")

In [ ]:
# ============================================================
# PAPER-STYLE MULTI-SECTOR LIGHT CURVE ANALYSIS
# FINAL CLEAN VERSION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.timeseries import (
    LombScargle,
    BoxLeastSquares
)

from wotan import flatten

# ============================================================
# USER INPUT
# ============================================================

TIC = 427393298

FILES = {
    6:  "tic 427393298_s_6_corrected.csv",
    32: "tic_427393298_s_32_corrected.csv",
    98: "tic_427393298_s_98_corrected.csv"
}

# ============================================================
# COLORS
# ============================================================

colors = {
    6: "black",
    32: "gray",
    98: "C0"
}

# ============================================================
# VISUAL OFFSETS
# ============================================================

offsets = {
    6: 0.00,
    32: 0.04,
    98: 0.08
}

# ============================================================
# LOAD DATA
# ============================================================

def load_sector_file(fname):

    df = pd.read_csv(fname)

    time_col = None
    flux_col = None

    for c in df.columns:

        cl = c.lower()

        if time_col is None and "time" in cl:
            time_col = c

        if flux_col is None and (
            "flux" in cl
            or "corr" in cl
            or "flat" in cl
        ):
            flux_col = c

    if time_col is None:
        raise ValueError("TIME column not found")

    if flux_col is None:
        raise ValueError("FLUX column not found")

    time = np.array(df[time_col])
    flux = np.array(df[flux_col])

    mask = np.isfinite(time) & np.isfinite(flux)

    time = time[mask]
    flux = flux[mask]

    # --------------------------------------------------------
    # sort
    # --------------------------------------------------------

    srt = np.argsort(time)

    time = time[srt]
    flux = flux[srt]

    return time, flux

# ============================================================
# ORBITAL PERIOD (BLS)
# ============================================================

# ============================================================
# ORBITAL PERIOD (ROBUST BLS)
# ============================================================

def get_orbital_period(time,
                       flux):

    flux = flux / np.nanmedian(flux)

    durations = np.linspace(
        0.02,
        0.2,
        20
    )

    bls = BoxLeastSquares(
        time,
        flux
    )

    periods = np.linspace(
        0.5,
        10,
        15000
    )

    result = bls.power(
        periods,
        durations
    )

    best = np.argmax(result.power)

    P_detected = result.period[best]

    # ========================================================
    # TEST P vs 2P
    # ========================================================

    P1 = P_detected
    P2 = 2 * P_detected

    # --------------------------------------------------------
    # fold scatter metric
    # --------------------------------------------------------

    def eclipse_metric(P):

        phase = (
            (time % P) / P
        )

        # sort
        srt = np.argsort(phase)

        phase = phase[srt]
        flux_s = flux[srt]

        # smooth scatter
        return np.nanstd(
            np.diff(flux_s)
        )

    m1 = eclipse_metric(P1)
    m2 = eclipse_metric(P2)

    # ========================================================
    # choose best
    # ========================================================

    if m2 < m1:

        P_orb = P2
        print("Using 2P harmonic correction")

    else:

        P_orb = P1

    # ========================================================
    # t0
    # ========================================================

    phase = (time % P_orb) / P_orb

    idx = np.argmin(flux)

    t0 = time[idx]

    return P_orb, t0

# ============================================================
# DETREND
# ============================================================

def detrend_flux(time,
                 flux,
                 P_orb,
                 sector,
                 eclipse_width=0.06):

    flux = flux / np.nanmedian(flux)

    # --------------------------------------------------------
    # dynamic WOTAN window
    # --------------------------------------------------------

    if sector in [6, 32]:

        window_length = 0.75

    else:

        window_length = 1.5

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flux)]

    phase = (
        ((time - t0) % P_orb) / P_orb
    ) - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width)
        |
        (np.abs(phase - 0.5) < eclipse_width)
        |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # --------------------------------------------------------
    # WOTAN
    # --------------------------------------------------------

    flat_flux, trend = flatten(
        time,
        flux,
        method='lowess',
        window_length=window_length,
        return_trend=True,
        mask=mask_eclipse
    )

    # --------------------------------------------------------
    # clean NaNs
    # --------------------------------------------------------

    mask = np.isfinite(flat_flux)

    time = time[mask]
    flux = flux[mask]
    flat_flux = flat_flux[mask]
    trend = trend[mask]
    mask_eclipse = mask_eclipse[mask]

    return (
        time,
        flux,
        flat_flux,
        trend,
        mask_eclipse
    )

# ============================================================
# ROTATION PERIOD
# ============================================================

def get_rotation_period(time,
                        flat_flux,
                        P_orb,
                        eclipse_width=0.06):

    # --------------------------------------------------------
    # eclipse mask
    # --------------------------------------------------------

    t0 = time[np.argmin(flat_flux)]

    phase = (
        ((time - t0) % P_orb) / P_orb
    ) - 0.5

    mask_eclipse = (
        (np.abs(phase) < eclipse_width)
        |
        (np.abs(phase - 0.5) < eclipse_width)
        |
        (np.abs(phase + 0.5) < eclipse_width)
    )

    # --------------------------------------------------------
    # clean rotation signal
    # --------------------------------------------------------

    tc = time[~mask_eclipse]
    fc = flat_flux[~mask_eclipse]

    # --------------------------------------------------------
    # Lomb-Scargle
    # --------------------------------------------------------

    freq, power = LombScargle(
        tc,
        fc
    ).autopower(
        minimum_frequency=1/20,
        maximum_frequency=1/0.3,
        samples_per_peak=20
    )

    periods = 1 / freq

    Prot = periods[np.argmax(power)]

    return Prot, tc, fc

# ============================================================
# ORBITAL PERIOD FROM FIRST SECTOR
# ============================================================

time0, flux0 = load_sector_file(
    FILES[6]
)

P_orb, t0 = get_orbital_period(
    time0,
    flux0
)

print(f"P_orb = {P_orb:.5f} d")

# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(16,10)
)

# ============================================================
# ORIGINAL
# ============================================================

ax1 = plt.subplot(221)

# ============================================================
# DETRENDED
# ============================================================

ax2 = plt.subplot(223)

# ============================================================
# ROTATIONAL FOLD
# ============================================================

ax3 = plt.subplot(222)

# ============================================================
# ORBITAL FOLD
# ============================================================

ax4 = plt.subplot(224)

# ============================================================
# LOOP SECTORS
# ============================================================

for sector in [6,32,98]:

    print(f"\nSector {sector}")

    # --------------------------------------------------------
    # load
    # --------------------------------------------------------

    time, flux = load_sector_file(
        FILES[sector]
    )

    # --------------------------------------------------------
    # shift time for visualization
    # --------------------------------------------------------

    time_plot = time - time.min()

    # --------------------------------------------------------
    # detrend
    # --------------------------------------------------------

    (
        time,
        flux,
        flat_flux,
        trend,
        mask_eclipse
    ) = detrend_flux(
        time,
        flux,
        P_orb,
        sector
    )

    # --------------------------------------------------------
    # rotation
    # --------------------------------------------------------

    Prot, tc, fc = get_rotation_period(
        time,
        flat_flux,
        P_orb
    )

    print(f"P_rot = {Prot:.5f}")

    # --------------------------------------------------------
    # offset
    # --------------------------------------------------------

    off = offsets[sector]

    # ========================================================
    # ORIGINAL LIGHT CURVE
    # ========================================================

    ax1.plot(
        time_plot,
        flux + off,
        '.',
        ms=2,
        alpha=0.5,
        color=colors[sector],
        label=f"Sector {sector}"
    )

    ax1.plot(
        time_plot,
        trend + off,
        color='red',
        lw=2
    )

    # ========================================================
    # DETRENDED
    # ========================================================

    ax2.plot(
        time_plot,
        flat_flux + off,
        '.',
        ms=2,
        alpha=0.5,
        color=colors[sector]
    )

    # ========================================================
    # ROTATIONAL PHASE
    # ========================================================

    phase_rot = (
        (tc % Prot) / Prot
    ) - 0.5

    phase_rot_plot = np.concatenate([
        phase_rot,
        phase_rot + 1
    ])

    flux_rot_plot = np.concatenate([
        fc,
        fc
    ])

    ax3.scatter(
        phase_rot_plot,
        flux_rot_plot + off,
        s=4,
        alpha=0.4,
        color=colors[sector],
        label=f"S{sector} | {Prot:.2f} d"
    )

    # ========================================================
    # ORBITAL PHASE
    # ========================================================

    phase_orb = (
        ((time - t0) % P_orb) / P_orb
    ) - 0.5

    phase_orb_plot = np.concatenate([
        phase_orb,
        phase_orb + 1
    ])

    flux_orb_plot = np.concatenate([
        flat_flux,
        flat_flux
    ])

    ax4.scatter(
        phase_orb_plot,
        flux_orb_plot + off,
        s=4,
        alpha=0.4,
        color=colors[sector],
        label=f"S{sector}"
    )

# ============================================================
# STYLING
# ============================================================

# ------------------------------------------------------------
# ORIGINAL
# ------------------------------------------------------------

ax1.set_title(
    "ORIGINAL"
)

ax1.set_xlabel("Shifted Time [days]")
ax1.set_ylabel("Normalized Flux")

ax1.legend()

# ------------------------------------------------------------
# DETRENDED
# ------------------------------------------------------------

ax2.set_title(
    "DETRENDED"
)

ax2.set_xlabel("Shifted Time [days]")
ax2.set_ylabel("Normalized Flux")

# ------------------------------------------------------------
# ROTATIONAL
# ------------------------------------------------------------

ax3.set_title(
    "Rotational Period"
)

ax3.set_xlabel("Folded Phase")
ax3.set_ylabel("Normalized Flux")

ax3.set_xlim(-0.5, 1.5)

ax3.legend()

# ------------------------------------------------------------
# ORBITAL
# ------------------------------------------------------------

ax4.set_title(
    f"Orbital Period\nP_orb = {P_orb:.3f} d"
)

ax4.set_xlabel("Folded Phase")
ax4.set_ylabel("Normalized Flux")

ax4.set_xlim(-0.5, 1.5)

ax4.legend()

# ============================================================
# FIX LIMITS
# ============================================================

ax1.set_ylim(0.85, 1.15)
ax2.set_ylim(0.85, 1.15)

ax3.set_ylim(0.85, 1.15)
ax4.set_ylim(0.85, 1.15)

# ============================================================
# GLOBAL TITLE
# ============================================================

plt.suptitle(
    f"TIC {TIC} — Light Curve Analysis",
    fontsize=28
)

plt.tight_layout()

plt.show()